# 11 — Extension modelling

This final model extension provides a focused example of more purposeful treatment of feature inputs. It reflects only a small part of the broader model-development evolution that occurred beyond the scope of this repository.

The original `TCN_starNLL` lineage is preserved. This notebook adds one bounded extension: causal upstream estimates for demand and regional direct radiation at the forecast origin, together with timestamp-aligned estimates of donor variability. Development remains confined to 2015–2019; the frozen 2020 evaluation is not opened or used.

## Purposeful treatment of availability

The extension separates data preparation from modelling. Before any TCN sequence is constructed, the upstream pipeline creates `x_estimated` and `x_estimated_sd` for every eligible timestamp using only earlier observations. The TCN never calculates donor averages or standard deviations.

For a sequence ending at forecast origin $t$, the physical channel is $[x_{t-n},\ldots,x_{t-1},x^{est}_t]$. The complete, timestamp-aligned `x_estimated_sd` series is supplied as an additional temporal feature. The estimate and its uncertainty therefore always refer to the same time.

## Causal donor rules

### Demand

- An ordinary weekday uses the same five-minute clock block and weekday in each of the previous three weeks.
- If one of those donor dates is a Queensland public holiday, that donor is replaced by the nearest Sunday at the same clock time.
- A non-holiday weekend uses the same weekend day and clock block in each of the previous three weeks.
- A current public holiday uses the same clock block on each of the three previous Sundays.
- The mean and sample standard deviation (`ddof=1`) are emitted only when all three donors exist. Initial missing values are retained.

### Direct radiation

Each regional estimate uses same-clock observations from the previous three calendar days. The historical radiation channel uses the last completed source hour; only its terminal value is replaced by `direct_radiation_estimated[t]`. Every donor predates the timestamp being estimated.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
import yaml

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.tcn_star_nll_extension_data import load_frame, prepare_fold, fold_endpoints
from src.tcn_star_nll_extension_model import TCNStarNLLCausalFeatureEstimates

CONFIG_PATH = ROOT / 'config/experiments/tcn_star_nll_causal_feature_estimates_validate_2019_seed42.yml'
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
config['model_name'], config['variant_name'], config['split']

('TCN_starNLL_causal_feature_estimates',
 'demand_direct_radiation_estimated_sd',
 {'train_start': '2015-01-01',
  'train_end_exclusive': '2019-01-01',
  'validation_start': '2019-01-01',
  'validation_end_exclusive': '2020-01-01',
  'seed': 42})

## Build and audit the upstream helpers

The next cell constructs the complete helper columns before any TCN dataset exists. It verifies the usable boundary, post-boundary completeness and the 2020 data ceiling.

In [2]:
frame = load_frame(
    ROOT / config['dataset'],
    ROOT / config['helper_sources']['demand'],
    ROOT / config['helper_sources']['direct_radiation'],
)
helper_columns = ['demand_estimated', 'demand_estimated_sd'] + [
    column for column in frame.columns if 'direct_radiation' in column
]
pd.Series({
    'first_usable_timestamp': frame.index.min(),
    'last_available_timestamp': frame.index.max(),
    'rows': len(frame),
    'helper_columns': len(helper_columns),
    'helper_missing_values': int(frame[helper_columns].isna().sum().sum()),
    'post_2020_rows': int((frame.index >= pd.Timestamp('2021-01-01')).sum()),
})

first_usable_timestamp      2015-01-22 00:05:00
last_available_timestamp    2020-12-31 23:25:00
rows                                     625241
helper_columns                               17
helper_missing_values                         0
post_2020_rows                                0
dtype: object

## Scaling choices

All transformations are fitted on 2015–2018 only. `demand_estimated` uses the original demand mean and standard deviation because it occupies the terminal position of that channel. `demand_estimated_sd` is divided by the demand training standard deviation without mean subtraction because dispersion has no location offset.

Regional direct-radiation observations and estimates use the same per-region training-fold minimum and maximum and are clipped to $[0,1]$. `direct_radiation_estimated_sd` is divided by the same regional range without centering. Using the physical feature's scale preserves the meaning of the uncertainty value and prevents validation information entering preprocessing.

In [3]:
train_start = pd.Timestamp('2015-01-01')
train_end = pd.Timestamp('2019-01-01')
validation_start = pd.Timestamp('2019-01-01')
validation_end = pd.Timestamp('2020-01-01')
arrays, preprocessing = prepare_fold(frame, train_start, train_end)
train_endpoints = fold_endpoints(frame, train_start, train_end)
validation_endpoints = fold_endpoints(frame, validation_start, validation_end)
pd.Series({
    'training_origins': len(train_endpoints),
    'validation_origins': len(validation_endpoints),
    'demand_tensor_width': arrays['demand'].shape[1],
    'regional_tensor_width': arrays['regional'].shape[2],
    'calendar_encoded_width': preprocessing['calendar_encoded_width'],
})

training_origins          414209
validation_origins        105114
demand_tensor_width            8
regional_tensor_width         12
calendar_encoded_width        53
dtype: int64

## Model construction and justification

The probabilistic backbone, 505-state lookback, causal convolutions, dilation schedule, dropout, fusion width, six horizons and bounded Student-t heads remain unchanged. Demand widens from 7 to 8 channels for aligned `demand_estimated_sd`. Each regional branch widens from 10 to 12 channels for direct radiation and aligned `direct_radiation_estimated_sd`. The estimate itself does not create another channel: it replaces only the terminal value of its corresponding physical channel.

In [4]:
model = TCNStarNLLCausalFeatureEstimates(calendar_dim=preprocessing['calendar_encoded_width'])
pd.Series({
    'parameters': sum(parameter.numel() for parameter in model.parameters()),
    'receptive_field_steps': model.receptive_field_steps,
    'demand_channels': model.DEMAND_CHANNELS,
    'regional_channels_per_region': model.REGIONAL_FEATURES,
})

parameters                      763794
receptive_field_steps             1009
demand_channels                      8
regional_channels_per_region        12
dtype: int64

## Exact authorized development run

The experiment trains on 2015–2018, validates on 2019 and uses seed 42. It retains AdamW, effective batch size 512, learning-rate reduction, early stopping and minimum-validation-NLL checkpoint selection. The frozen 2020 gate remains untouched. Full training is disabled by default inside the notebook to prevent accidental reruns.

In [5]:
RUN_FULL_TRAINING = False
command = [
    sys.executable,
    str(ROOT / 'tools/run_tcn_star_nll_causal_feature_estimates_validate_2019.py'),
    '--authorise-full-run',
]
print('python tools/run_tcn_star_nll_causal_feature_estimates_validate_2019.py --authorise-full-run')
if RUN_FULL_TRAINING:
    import subprocess
    subprocess.run(command, cwd=ROOT, check=True)

python tools/run_tcn_star_nll_causal_feature_estimates_validate_2019.py --authorise-full-run


## Retained 2019 ensemble result and interpretation boundary

The final extension comparison now includes the completed no-SD three-seed ensemble. All neural results use the same 2015–2018 training period and 2019 validation year. Each no-SD ensemble member uses its own minimum-validation-NLL checkpoint: 42:10, 142:9, 242:10. The ensemble is an equal-weight mixture of the three Student-t predictive distributions; its point forecast is the arithmetic mean of their locations. No ensemble weight was fitted.

| Candidate rank by MAE | Result | Evidence scope | Validation NLL | MAE (MW) | RMSE (MW) |
|---:|---|---|---:|---:|---:|
| 1 | `TCN_starNLL_causal_feature_estimates_no_sd` | Three-seed Student-t ensemble | 0.718355 | 37.583 | 49.329 |
| Supporting member | `TCN_starNLL_causal_feature_estimates_no_sd`, seed 42 | Single minimum-NLL checkpoint | 0.734999 | 38.091 | 49.973 |
| 2 | `TCN_starNLL_causal_feature_estimates`, seed 42 | Single minimum-NLL checkpoint | 0.736293 | 38.203 | 50.165 |
| 3 | `TCN_starNLL_direct_radiation`, seed 42 | Single minimum-NLL checkpoint | 0.737551 | 38.287 | 50.301 |
| External benchmark | AEMO P5MIN | Aligned 2019 point forecasts | — | 47.119 | 61.315 |

Relative to its retained seed-42 member, the no-SD ensemble lowers standardized NLL by 0.016644, MAE by 0.508 MW and RMSE by 0.644 MW. The top two **distinct RR model candidates** under the existing MAE ranking are therefore the no-SD ensemble and the SD-inclusive extension. Their evidence scopes differ—the first is a three-seed ensemble and the second remains a single-seed result—so the table states that distinction directly rather than implying equal replication.

AEMO remains an external point-forecast benchmark and has no probabilistic NLL. This development extension does not alter the original model-selection path, does not supersede the published `TCN_starNLL` result, and does not open the fixed 2020 chronological evaluation.

In [6]:
comparison_path = ROOT / 'outputs/11_extension_modelling/model_comparison_2019_with_no_sd_ensemble.csv'
comparison = pd.read_csv(comparison_path, keep_default_na=False)
display(comparison)

,candidate_rank_by_mae,result_label,model,comparison_role,evidence_scope,train_period,validation_year,seed_scope,selected_epochs_by_seed,validation_nll,validation_mae_mw,validation_rmse_mw
0,1,No-SD causal-estimate extension — three-seed e...,TCN_starNLL_causal_feature_estimates_no_sd,final extension ensemble,equal-weight Student-t mixture,2015-2018,2019,"42, 142, 242","42:10, 142:9, 242:10",0.718355081472699,37.582864,49.329142
1,supporting member,No-SD causal-estimate extension — seed 42,TCN_starNLL_causal_feature_estimates_no_sd,retained single-seed member evidence,minimum-validation-NLL checkpoint,2015-2018,2019,42,42:10,0.7349993985174708,38.090700,49.972866
2,2,SD-inclusive causal-estimate extension — seed 42,TCN_starNLL_causal_feature_estimates,second-ranked distinct RR model candidate,minimum-validation-NLL checkpoint,2015-2018,2019,42,42:11,0.7362934529882276,38.203182,50.164644
3,3,Direct-radiation model — seed 42,TCN_starNLL_direct_radiation,previous non-extension champion,minimum-validation-NLL checkpoint,2015-2018,2019,42,42:7,0.7375509840650464,38.287194,50.300569
4,external benchmark,AEMO P5MIN external benchmark,AEMO P5MIN external benchmark,external point-forecast benchmark,aligned 2019 point forecasts,not applicable,2019,not applicable,not applicable,not available,47.119002,61.314686


## Final held-out 2020 result

### Headline: probabilistic six-horizon forecasting with 22.44% lower 30-minute MAE than AEMO P5MIN

The 2019 evidence above selected the three-seed `TCN_starNLL_noSD` ensemble. Only that selected model proceeded to the final study-period evaluation. Its seed 42, 142, and 242 members were refitted on 2015–2019 for exactly 10 epochs, frozen, and combined as an equal-weight Student-t mixture before 2020 was scored. The final comparator is the published AEMO P5MIN forecast at the equivalent 30-minute horizon; other development candidates are not ranked on 2020.

The model also produces a complete predictive distribution at every 5, 10, 15, 20, 25, and 30-minute horizon. Each member predicts Student-t location, scale, and degrees of freedom, and the three densities are combined as an equal-weight mixture. This supports prediction intervals, tail-risk measures, and threshold probabilities in addition to the point forecast.

**Illustrative example.** If current demand is 7,000 MW and a 30-minute head has a change location of +120 MW, scale of 60 MW, and 5 degrees of freedom, the point forecast is 7,120 MW. The same Student-t head assigns approximately 12% probability to demand exceeding 7,200 MW (`change > 200 MW`). These values are explanatory only and are not a selected evaluation row.

| Rank | Model | Role | Forecast origins | 30-minute MAE (MW) | 30-minute RMSE (MW) | Bias (MW) | MAE gain vs AEMO |
|---:|---|---|---:|---:|---:|---:|---:|
| 1 | `TCN_starNLL_noSD` three-seed ensemble | 2019-selected champion | 105,400 | **47.355** | **64.308** | 2.672 | **22.44%** |
| 2 | AEMO P5MIN external benchmark | Published external benchmark | 105,400 | 61.056 | 81.256 | -1.095 | Baseline |

The absolute MAE reduction is **13.701 MW**. The relative gain is calculated as `(61.056 - 47.355) / 61.056 × 100` on the 105,400 common forecast origins.

In [7]:
final_comparison_path = ROOT / 'outputs/10_final_2020_evaluation/endpoint_30m_model_comparison_2020.csv'
final_comparison = pd.read_csv(final_comparison_path)
display(final_comparison.round(4))

,rank_by_mae,model,role,forecast_origins,mae_30m_mw,rmse_30m_mw,bias_30m_mw,mae_gain_vs_aemo_percent
0,1,TCN_starNLL_noSD three-seed ensemble,2019-selected champion,105400,47.3549,64.3077,2.6718,22.4402
1,2,AEMO P5MIN external benchmark,external benchmark,105400,61.0560,81.2563,-1.0950,0.0000
